<a href="https://colab.research.google.com/github/suhaant/ML-Housing-Valuation-Model/blob/main/RANDFOR_Housing_Prices_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("WakeCountyHousing.csv")

# Drop missing
df = df[df['Physical_City'].notna()].dropna()

# Keep only homes built after 2005
df = df[df['Year_Built'] > 2005]

# Optional: If you have Sale_Year, filter recent
# df = df[df['Sale_Year'] >= 2020]

# Filter price range (remove extreme outliers)
df = df[(df['Total_Sale_Price'] > 100000) & (df['Total_Sale_Price'] < 2_500_000)]


In [ ]:
# Clean numeric columns
df['Bath'] = df['Bath'].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)
df['Heated_Area'] = pd.to_numeric(df['Heated_Area'], errors='coerce')
df['Deeded_Acreage'] = pd.to_numeric(df['Deeded_Acreage'], errors='coerce')
df['Physical_Zip'] = df['Physical_Zip'].astype(str)

# Create new features
df['Price_per_sqft'] = df['Total_Sale_Price'] / df['Heated_Area']
df['Large_Lot'] = (df['Deeded_Acreage'] > 1).astype(int)  # Flag for >1 acre
df['Lot_SqFt'] = df['Deeded_Acreage'] * 43560  # 1 acre = 43,560 sqft

# --- Add Physical_City as a categorical feature ---
df['Physical_City'] = df['Physical_City'].astype(str)

# --- OPTIONAL: Add HOA fee if available ---
# If you have an HOA_fee column, clean and convert:
# df['HOA_Fee'] = pd.to_numeric(df['HOA_Fee'], errors='coerce').fillna(0)

# --- Compute average $/sqft per ZIP ---
df['Price_per_sqft'] = df['Total_Sale_Price'] / df['Heated_Area']
zip_avg_price_per_sqft = df.groupby('Physical_Zip')['Price_per_sqft'].mean()
df['Zip_Avg_Price_per_sqft'] = df['Physical_Zip'].map(zip_avg_price_per_sqft)


# Keep only top 25 ZIPs
top_zips = df['Physical_Zip'].value_counts().nlargest(25).index
df = df[df['Physical_Zip'].isin(top_zips)].copy()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define features and target
features = [
    'Heated_Area', 'Bath', 'Deeded_Acreage', 'Lot_SqFt', 'Year_Built',
    'Large_Lot', 'Zip_Avg_Price_per_sqft', 'Physical_City', 'Physical_Zip'
]
X = df[features]
y = np.log1p(df['Total_Sale_Price'])  # log-transform target

# Preprocessing
numerical = ['Heated_Area', 'Bath', 'Deeded_Acreage', 'Lot_SqFt', 'Year_Built', 'Large_Lot', 'Zip_Avg_Price_per_sqft']
categorical = ['Physical_City', 'Physical_Zip']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set up pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)

# Predict and un-log
y_pred_log = model_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)


In [ ]:
# Calculate metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Print results
print("📊 Improved Model Evaluation:")
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
from datetime import datetime

def predict_custom_future_price(heated_area, bath, acreage, year_built, zip_code, city):
    lot_sqft = acreage * 43560
    large_lot = int(acreage > 1)
    # Get ZIP avg $/sqft (from training data)
    zip_avg_price = zip_avg_price_per_sqft.get(str(zip_code), zip_avg_price_per_sqft.mean())

    input_df = pd.DataFrame([{
        'Heated_Area': heated_area,
        'Bath': bath,
        'Deeded_Acreage': acreage,
        'Lot_SqFt': lot_sqft,
        'Year_Built': year_built,
        'Large_Lot': large_lot,
        'Zip_Avg_Price_per_sqft': zip_avg_price,
        'Physical_City': str(city),
        'Physical_Zip': str(zip_code)
    }])
    predicted_log_price = model_pipeline.predict(input_df)[0]
    predicted_price = np.expm1(predicted_log_price)
    return predicted_price

def predict_future_price(heated_area, bath, acreage, year_built, zip_code, city, future_year):
    annual_appreciation_rate = 0.038  # 3.8% annual appreciation
    current_price = predict_custom_future_price(heated_area, bath, acreage, year_built, zip_code, city)
    years_out = future_year - datetime.now().year
    if years_out < 0:
        return "❌ Future year must be after current year."
    future_price = current_price * ((1 + annual_appreciation_rate) ** years_out)
    return f"🏡 Estimated Future Price in {future_year}: ${future_price:,.2f}"


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Input widgets
heated_input = widgets.IntText(value=2500, description='Sq Ft:')
bath_input = widgets.FloatText(value=3.0, description='Baths:')
lot_input = widgets.FloatText(value=0.5, description='Acres:')
year_built_input = widgets.IntText(value=2015, description='Year Built:')
zip_input = widgets.Dropdown(options=sorted(df['Physical_Zip'].unique()), description='ZIP Code:')
city_input = widgets.Dropdown(options=sorted(df['Physical_City'].unique()), description='City:')
future_year_input = widgets.IntText(value=2030, description='Future Year:')
button = widgets.Button(description="Predict Future Price")
output = widgets.Output()

# Click handler
def on_button_click(b):
    with output:
        output.clear_output()
        result = predict_future_price(
            heated_input.value,
            bath_input.value,
            lot_input.value,
            year_built_input.value,
            zip_input.value,
            city_input.value,
            future_year_input.value
        )
        print(result)

button.on_click(on_button_click)

# Display UI
display(
    heated_input, bath_input, lot_input,
    year_built_input, zip_input, future_year_input,
    button, output
)
